In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
train = pd.read_csv('labels_day_train_features.csv', sep = ';').drop('date', axis=1)
test = pd.read_csv('labels_day_test_features.csv', sep = ';').drop('date', axis=1)

In [4]:
train = train.drop(columns=['date'], errors="ignore")
test = test.drop(columns=['date'], errors="ignore")

In [5]:
def wape(y_true, y_pred):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    return f'{(1 - (abs(y_true - y_pred).sum() / abs(y_true).sum())):.4f}'

In [7]:
nums_col = ["temperature_2m (°C)", "apparent_temperature (°C)",
            "precipitation (mm)", "rain (mm)", "snowfall_mm",
            "snow_depth_mm", "pressure_msl (hPa)",
            "wind_speed_10m (km/h)", "wind_gusts_10m (km/h)",
            ]


from pathlib import Path
import joblib
from sklearn.preprocessing import MinMaxScaler

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
df_all = pd.concat([train, test], ignore_index=True)

scaler = MinMaxScaler()
df_all[nums_col] = scaler.fit_transform(df_all[nums_col])
# train[nums_col] = scaler.fit_transform(train[nums_col])
# test[nums_col] = scaler.transform(test[nums_col])
joblib.dump(scaler, ARTIFACTS_DIR / "minmax_scaler.joblib")


['artifacts\\minmax_scaler.joblib']

In [8]:
train_routes = sorted(df_all['route'].dropna().unique())
# train_routes = sorted(train['route'].dropna().unique())

test_datasets_by_route = {
    route: test[test['route'] == route].copy()
    for route in train_routes
}

# datasets_by_route = {
#     route: train[train['route'] == route].copy()
#     for route in train_routes
# }

datasets_by_route = {
    route: df_all[df_all['route'] == route].copy()
    for route in train_routes
}

# for route in train_routes:
#     print(f"?????? {route}: train={len(datasets_by_route[route])}, test={len(test_datasets_by_route[route])}")


In [ ]:
import matplotlib.pyplot as plt
import joblib
import catboost as ctb

nums_all = ["temperature_2m (°C)", "apparent_temperature (°C)",
            "precipitation (mm)", "rain (mm)", "snowfall_mm",
            "snow_depth_mm", "pressure_msl (hPa)",
            "wind_speed_10m (km/h)", "wind_gusts_10m (km/h)",
            'wind_direction_sin', 'wind_direction_cos',
            'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos',
            'month_sin', 'month_cos', 'dayofyear_sin',	'dayofyear_cos',
            "boardings", "route"
            ]

cat_cols = [i for i in train.columns.to_list() if i not in nums_all]
models = {}
midlle = 0

for route in train_routes:
    Y = datasets_by_route[route]["boardings"]
    X = datasets_by_route[route].drop(columns=["boardings", "route"])
    Y_test = test_datasets_by_route[route]["boardings"]
    X_test = test_datasets_by_route[route].drop(columns=["boardings", "route"])

    model = ctb.CatBoostRegressor(
        loss_function="MAE",
        cat_features = cat_cols,
        eval_metric="MAE",
        iterations=500,
        learning_rate=0.05,
        depth=5,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        # loss_function="MAE",
        # cat_features = cat_cols,
        # eval_metric="MAE",
        # iterations=500,
        # learning_rate=0.05,
        # depth=5,
        # l2_leaf_reg=5,
        # random_seed=42,
        # verbose=False,
        # allow_writing_files=False,
    )

    model.fit(
        X, Y,
        verbose=False,
    )
   

    models[route] = model
    model_path = ARTIFACTS_DIR / f"catboost_route_{route}.joblib"
    joblib.dump(model, model_path)

    y_pred = model.predict(X_test)
    midlle += float(wape(Y_test, y_pred))
    print(f"route {route}: {wape(Y_test, y_pred)};")

print(f'wape: {midlle / 9}')

# route 1: 0.8543;
# route 7: 0.7657;
# route 11: 0.8789;
# route 12: 0.9051;
# route 17: 0.9129;
# route 25: 0.7175;
# route 26: 0.8083;
# route 28: 0.8281;
# route 50: 0.7402;

route 1: 0.8804;
route 7: 0.8407;
route 11: 0.8885;
route 12: 0.9120;
route 17: 0.9207;
route 25: 0.8334;
route 26: 0.8891;
route 28: 0.8122;
route 50: 0.8516;
wape: 0.8698444444444444
